# Endothelial subset: scVI and PCA workflows

Project: **Characterisation of endothelial cells in progressive fatty liver disease in humans.**

This notebook retains sparse data import, count/log-normalized layers, scVI model fitting and latent-space clustering, a separate PCA route, exploratory marker routines, barcode filtering and group-wise plotting code. The original scVI setup had no batch key; its latent representation is not evidence that batch correction was performed.

These are edited source excerpts from my analysis notebooks. All saved outputs, execution counts, metadata, feature panels, group labels and findings have been removed. No data or figures are included. Private inputs are unavailable here, so these excerpts have not been rerun and do not reproduce the study.

Execution is disabled by default. Plotting calls remain as code only; any future output belongs in the private data directory. Algorithm settings are retained from the source and need reassessment for other inputs. Generic metadata keys stand in for private column names. Some routines use tutorial/shared analysis patterns, as in the original notebooks.

In [ ]:
from pathlib import Path

RUN_PRIVATE_ANALYSIS = False
RUN_EXTERNAL_ENRICHMENT = False
INPUT_FORMAT = "matrix"
PRIVATE_DIR = Path("data/private")
OUTPUT_DIR = PRIVATE_DIR / "results"
BATCH_KEY = None  # Original scVI call did not supply a batch key.
selected_clusters = []  # Private, locally selected IDs; no study IDs supplied.
features_to_review = []  # Private feature panel omitted.
approved_enrichment_genes = []  # Only externally approved lists may be submitted.


## Imports and display settings

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Required packages
    import numpy as np
    import pandas as pd
    import scanpy as sc
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib import rcParams

    sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)

    sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300, frameon=True, vector_friendly=True, fontsize=20)
    rcParams['figure.figsize'] = 8, 8
    plt.rcParams.update({'lines.markersize' : '15'})



    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    sc.settings.figdir = OUTPUT_DIR


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300, frameon=True, vector_friendly=True, fontsize=20)
    rcParams['figure.figsize'] = 8, 8
    import matplotlib.pyplot as plt
    plt.rcParams.update({'lines.markersize' : '15'})

    palette = [ '#FFFF00', '#1CE6FF', '#FF34FF', '#FF4A46',
        '#008941', '#006FA6', '#A30059', '#FFDBE5',
        '#7A4900', '#0000A6', '#63FFAC', '#B79762',
        '#004D43', '#8FB0FF', '#997D87', '#5A0007',
        '#809693', '#6A3A4C', '#1B4400', '#4FC601',
        '#3B5DFF', '#4A3B53', '#FF2F80', '#61615A',
        '#BA0900', '#6B7900', '#00C2A0', '#FFAA92',
        '#FF90C9', '#B903AA', '#D16100', '#DDEFFF',
        '#000035', '#7B4F4B', '#A1C299', '#300018',
        '#0AA6D8', '#013349', '#00846F', '#372101',
        '#FFB500', '#C2FFED', '#A079BF', '#CC0744',
        # new additions
        '#3CB44B', '#FFE119', '#4363D8', '#F58231',
        '#911EB4', '#42D4F4', '#BFEF45', '#469990',
        '#DCBEFF', '#9A6324', '#FFFAC8', '#800000',
        '#AAFFC3', '#FFD8B1', '#000075', '#B5BD68',
        '#8ABEB7', '#CC6666', '#F0C674', '#C5C8C6',
        '#DE935F', '#A3685A', '#4ECDC4', '#FF6B6B',
        '#81D4FA', '#F06292', '#DCE775', '#CE93D8',
        '#E6EE9C', '#26C6DA', '#66BB6A', '#FFA726',
        '#42A5F5', '#AB47BC']


## Private input and optional QC

Choose the Matrix Market or h5ad entry point. Inputs must have raw, nonnegative integer counts and aligned cell/feature tables. The selection of cells is not disclosed.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    if INPUT_FORMAT == "h5ad":
        adata = sc.read_h5ad(PRIVATE_DIR / "input_subset.h5ad")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    if INPUT_FORMAT == "matrix":
        # Construct AnnData from locally exported count and metadata files.
        from scipy.io import mmread

        # 1. Load the matrix (Transpose from R's Genes x Cells to Python's Cells x Genes)
        counts = mmread(PRIVATE_DIR / "subset_counts.mtx").T.tocsr()

        # 2. Load metadata and genes
        obs = pd.read_csv(PRIVATE_DIR / "subset_metadata.csv", index_col=0)
        genes = pd.read_csv(PRIVATE_DIR / "subset_genes.csv")

        # 3. Build AnnData
        adata = sc.AnnData(X=counts, obs=obs)
        adata.var_names = genes.iloc[:, 0].astype(str).values


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    count_values = adata.X.data if hasattr(adata.X, "tocsr") else np.asarray(adata.X).ravel()
    if not (np.isfinite(count_values).all() and (count_values >= 0).all() and np.allclose(count_values, np.rint(count_values), rtol=0, atol=1e-8)):
        raise ValueError("This input must contain raw count data; supply the count object locally.")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # ##Check QC- based on scanpy tutorial
    # # Broad QC feature prefixes, adjusted to the disclosed human context.
    # adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # # ribosomal genes
    # adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # # hemoglobin genes
    # adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

    # sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

    # sc.pl.violin(
    #     adata,
    #     ["pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    #     jitter=0.4,
    #     multi_panel=True,
    # )

    # sc.pl.violin(
    #     adata,
    #     ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    #     jitter=0.4,
    #     multi_panel=True,
    # )
    pass  # Optional QC recipe above is intentionally commented.


## scVI model and latent representation

The HVG step annotates genes with `subset=False`; the following source model uses the full AnnData object. Count data are supplied through the `counts` layer. An optional batch key is shown but remains unset, matching the original executed call.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Keep raw counts in a layer — scVI requires raw (integer) counts
    adata.layers["counts"] = adata.X.copy()


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # ── 3. Preprocessing ──────────────────────────────────────────────────────────
    #Normalize & log-transform for non-scVI analyses (UMAP labels, etc.)
    # 2. Normalize total counts per cell to 10,000
    sc.pp.normalize_total(adata, target_sum=1e4)

    # 3. Log1p transform (log(x + 1))
    sc.pp.log1p(adata)

    # 4. Save the result to a new layer
    adata.layers["lognorm"] = adata.X.copy()

    # 5. (Optional) restore raw counts to X for compatibility with scVI
    adata.X = adata.layers["counts"]


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Annotate highly variable genes; subset=False keeps the full object.
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=3000,          
        layer="counts",       # use raw counts for HVG selection
        flavor="seurat_v3",   # recommended for count data
        #batch_key="batch_id",    # correct for batch during HVG selection
        subset=False,            # keep the complete feature set
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import scvi
    import scanpy as sc

    # 1. Setup the AnnData
    # Note: Ensure adata.layers["counts"] contains your raw integers
    scvi.model.SCVI.setup_anndata(
        adata, 
        layer="counts", 
        batch_key=BATCH_KEY
    )

    # 2. Initialize the model 
    # Retain the original model dimensions.
    model = scvi.model.SCVI(
        adata, 
        n_layers=2, 
        n_latent=30, 
        gene_likelihood="nb" # Negative Binomial is standard for counts
    )

    # 3. Train the model
    model.train()

    # 4. Extract the results
    adata.obsm["X_scVI"] = model.get_latent_representation()


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Plot training loss
    train_elbo = model.history["elbo_train"]
    #val_elbo   = model.history["elbo_validation"]
 
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(train_elbo.index, train_elbo["elbo_train"],   label="Train ELBO")
    #ax.plot(val_elbo.index,   val_elbo["elbo_validation"], label="Val ELBO")
    ax.set_xlabel("Epoch"); ax.set_ylabel("ELBO"); ax.legend()
    ax.set_title("scVI Training Loss")
    # fig.tight_layout()
    # plt.savefig(OUTPUT_DIR / "private_plot.pdf", dpi=150)
    # plt.close()
 
    # Save trained model


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # ── 7. Dimensionality Reduction & Clustering ─────────────────────────────────
    # Neighbors on scVI latent space
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=30)  # try 10–50
 
    # UMAP
    sc.tl.umap(adata) 
 
    # Leiden clustering at multiple resolutions
    sc.tl.leiden(adata, resolution=1, flavor="igraph", n_iterations=2)
    #for res in [0.8, 1.0, 1.2]:
    #    sc.tl.leiden(adata, flavor="igraph", n_iterations=2, resolution=res, key_added=f"leiden_scvi_{res}")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pl.umap(adata, wspace=1.2, color=["leiden"], size=20, palette=palette)


## PCA comparison

The PCA route restores raw counts before normalization. It overwrites the active neighbor graph, UMAP and Leiden labels, as in the exploratory source notebook. Subsequent marker code uses this active clustering.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # ── 3. Preprocessing ──────────────────────────────────────────────────────────
    # Keep raw counts in a layer — scVI requires raw (integer) counts
    adata.X = adata.layers['counts'].copy()

    #Normalize & log-transform for non-scVI analyses (UMAP labels, etc.)
    # 2. Normalize total counts per cell to 10,000
    sc.pp.normalize_total(adata, target_sum=1e4)

    # 3. Log1p transform (log(x + 1))
    sc.pp.log1p(adata)

    # 4. Save the result to a new layer
    adata.layers["lognorm"] = adata.X.copy()

    # 5. (Optional) restore raw counts to X for compatibility with scVI
    #adata.X = adata.layers["counts"]


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Annotate highly variable genes; subset=False keeps the full object.
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=3000,          
        layer="counts",       # use raw counts for HVG selection
        flavor="seurat_v3",   # recommended for count data
        #batch_key="batch_id",    # correct for batch during HVG selection
        subset=False,            # keep the complete feature set
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    ###Using PCA instead of scvi
    sc.tl.pca(adata)
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=30)  # try 10–50
    sc.tl.leiden(adata, resolution=1.2, flavor="igraph", n_iterations=2)
    sc.tl.umap(adata)  # min_dist ~0.1–0.5


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pl.umap(adata, wspace=0.6, color=["leiden"], size=20, palette=palette)


## Exploratory cluster rankings

These are descriptive cell-level rankings, not donor-level differential-expression tests. Tutorial and shared-analysis patterns used in the source are retained. No ranked genes or selected cluster IDs are included.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    #Tutorial based method
    # Obtain cluster-specific differentially expressed genes
    sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon", layer="lognorm", use_raw=False)
    sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", layer="lognorm", use_raw=False, standard_scale="var", n_genes=20, save="_private_analysis.pdf")
    if selected_clusters:
        dc_cluster_genes = sc.get.rank_genes_groups_df(adata, group=selected_clusters[0]).head(20)["names"]
        sc.pl.umap(
            adata,
            color=[*dc_cluster_genes, "leiden"],
            layer="lognorm", use_raw=False,
            legend_loc="on data",
            frameon=False,
            ncols=3,
        )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Adapted shared-analysis pattern used in the original notebook.
    sc.tl.rank_genes_groups(adata, 'leiden', method='t-test', layer="lognorm", use_raw=False)
    sc.pl.rank_genes_groups(adata, n_genes=50, sharey=False, save="_private_analysis.pdf")

    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    adata_deg = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'pvals', 'scores']})
    adata_deg.to_csv(OUTPUT_DIR / "cluster_rankings.csv")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Get top 20 DEGs per cluster by score (score ranking; not fold-change ranking) similar to seurat in R
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names

    # build a dataframe with names and scores
    deg_df = pd.DataFrame({
        'gene':    [gene  for group in groups for gene  in result['names'][group][:20]],
        'score':   [score for group in groups for score in result['scores'][group][:20]],
        'cluster': [group for group in groups for _    in result['names'][group][:20]]
    })

    # top 20 per cluster by score
    top20 = (
        deg_df
        .groupby('cluster')
        .apply(lambda x: x.nlargest(20, 'score'))
        .reset_index(drop=True)
    )

    top20_genes = top20['gene'].tolist()

    print(top20)
    print(f"\nTotal genes for heatmap: {len(top20_genes)}")

    # Visualize using plots in section below


### Additional exploratory ranking variants

These alternative recipes are retained as code patterns. They do not establish biological specificity. Thresholds and selected clusters require local review; one commented source variant was not used in the study.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import pandas as pd
    import scanpy as sc

    # Step 1: rebuild full DEG dataframe
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names

    deg_df = pd.DataFrame({
        'gene':     [gene  for group in groups for gene  in result['names'][group]],
        'score':    [score for group in groups for score in result['scores'][group]],
        'pval_adj': [pval  for group in groups for pval  in result['pvals_adj'][group]],
        'cluster':  [group for group in groups for _     in result['names'][group]]
    })

    # Step 2: filter by adjusted pvalue
    top50_per_cluster = (
        deg_df[deg_df['pval_adj'] < 0.05]
        .groupby('cluster')
        .apply(lambda x: x.nlargest(50, 'score'))
        .reset_index(drop=True)
    )

    # Step 3: keep genes appearing in exactly 1 cluster
    gene_counts    = top50_per_cluster['gene'].value_counts()
    unique_genes   = gene_counts[gene_counts == 1].index
    unique_markers = top50_per_cluster[top50_per_cluster['gene'].isin(unique_genes)]

    # Step 4: top 10 unique genes per cluster
    top10_unique = (
        unique_markers
        .groupby('cluster')
        .apply(lambda x: x.nlargest(10, 'score'))
        .reset_index(drop=True)
    )

    print(top10_unique)
    print(f"\nClusters represented: {top10_unique['cluster'].nunique()}")
    print(f"Total unique genes:   {top10_unique['gene'].nunique()}")

    # Step 5: group genes by cluster for ordered plotting
    genes_by_cluster = (
        top10_unique
        .groupby('cluster')['gene']
        .apply(list)
        .to_dict()
    )

    # Step 6: dotplot
    sc.pl.dotplot(
        adata,
        var_names=genes_by_cluster,
        groupby='leiden',
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(30, 10),
        show=False,
        save="_private_analysis.pdf"
    )

    # Step 7: matrixplot
    sc.pl.matrixplot(
        adata,
        var_names=genes_by_cluster,
        groupby='leiden',
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(30, 10),
        show=False,
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS and selected_clusters:
    # Optional locally selected clusters; study selection omitted.
    #run 1-vs-rest but only for problem clusters, with stricter filters
    problem_clusters = selected_clusters

    sc.tl.rank_genes_groups(
        adata,
        groupby='leiden',
        key_added="selected_cluster_rankings",
        groups=problem_clusters,    # only run for problem clusters
        method='wilcoxon',
        layer='lognorm',
        use_raw=False
    )

    result = adata.uns['selected_cluster_rankings']

    # build dataframe with logfc filter
    deg_problem = pd.DataFrame({
        'gene':     [gene  for group in problem_clusters for gene  in result['names'][group]],
        'score':    [score for group in problem_clusters for score in result['scores'][group]],
        'logfc':    [lfc   for group in problem_clusters for lfc   in result['logfoldchanges'][group]],
        'pval_adj': [pval  for group in problem_clusters for pval  in result['pvals_adj'][group]],
        'cluster':  [group for group in problem_clusters for _     in result['names'][group]]
    })

    # stricter filters — high logfc and significance
    deg_problem_filtered = deg_problem[
        (deg_problem['pval_adj'] < 0.005) &
        (deg_problem['logfc'] > 1.0)        # increase this if still too many shared genes
    ]

    # top 10 per cluster
    top10_problem = (
        deg_problem_filtered
        .groupby('cluster')
        .apply(lambda x: x.nlargest(10, 'logfc'))   # rank by logfc not score
        .reset_index(drop=True)
    )

    print(top10_problem)
    print(top10_problem['cluster'].value_counts())


In [ ]:
if RUN_PRIVATE_ANALYSIS and selected_clusters:
    # combine with easy cluster markers from before
    other_clusters_markers = top10_unique[~top10_unique['cluster'].isin(problem_clusters)]

    final_markers = pd.concat([
        other_clusters_markers[['gene', 'cluster']],
        top10_problem[['gene', 'cluster']]
    ], ignore_index=True)

    # sort by cluster
    final_markers['cluster'] = final_markers['cluster'].astype(int)
    final_markers = final_markers.sort_values('cluster')
    final_markers['cluster'] = final_markers['cluster'].astype(str)

    # group for plots
    genes_by_cluster_final = (
        final_markers
        .groupby('cluster')['gene']
        .apply(list)
        .to_dict()
    )

    cluster_order = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))

    # dotplot
    sc.pl.dotplot(
        adata,
        var_names=genes_by_cluster_final,
        groupby='leiden',
        categories_order=cluster_order,
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(35, 12),
        show=False,
        save="_private_analysis.pdf"
    )

    # matrixplot
    sc.pl.matrixplot(
        adata,
        var_names=genes_by_cluster_final,
        groupby='leiden',
        categories_order=cluster_order,
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(35, 12),
        show=False,
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import pandas as pd
    import scanpy as sc

    # rebuild full DEG dataframe with all stats
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names

    deg_df = pd.DataFrame({
        'gene':    [gene  for group in groups for gene  in result['names'][group]],
        'score':   [score for group in groups for score in result['scores'][group]],
        'pval':    [pval  for group in groups for pval  in result['pvals'][group]],
        'pval_adj':[pval  for group in groups for pval  in result['pvals_adj'][group]],
        'cluster': [group for group in groups for _     in result['names'][group]]
    })

    # Heuristic score difference: cluster score minus mean score in other clusters.
    # This descriptive heuristic does not establish biological specificity.

    all_clusters = deg_df['cluster'].unique()

    specificity_dfs = []

    for cluster in all_clusters:
        # scores in this cluster
        this = deg_df[deg_df['cluster'] == cluster][['gene','score']].rename(columns={'score':'score_this'})
    
        # mean score of same gene in all OTHER clusters
        others = (deg_df[deg_df['cluster'] != cluster]
                  .groupby('gene')['score']
                  .mean()
                  .reset_index()
                  .rename(columns={'score':'score_others'}))
    
        merged = this.merge(others, on='gene')
        merged['specificity'] = merged['score_this'] - merged['score_others']  # higher = more unique
        merged['cluster'] = cluster
        specificity_dfs.append(merged)

    specificity_df = pd.concat(specificity_dfs)

    # top 10 most specific genes per cluster
    top10_specific = (
        specificity_df
        .groupby('cluster')
        .apply(lambda x: x.nlargest(10, 'specificity'))
        .reset_index(drop=True)
    )

    print(top10_specific)
    unique_genes_final = top10_specific['gene'].tolist()


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # filter DEGs by fold change and fraction of cells expressing
    sc.tl.filter_rank_genes_groups(
        adata,
        min_in_group_fraction=0.25,   # gene must be expressed in >25% of cluster cells
        max_out_group_fraction=0.5,   # gene expressed in <50% of other cells = more specific
        min_fold_change=1.5           # minimum fold change
    )

    # get filtered results
    result_filtered = adata.uns['rank_genes_groups_filtered']
    groups = result_filtered['names'].dtype.names

    deg_filtered = pd.DataFrame({
        'gene':    [gene  for group in groups for gene  in result_filtered['names'][group] if pd.notna(gene) and gene != ''],
        'score':   [score for group in groups for gene, score in zip(result_filtered['names'][group], result_filtered['scores'][group]) if pd.notna(gene) and gene != ''],
        'cluster': [group for group in groups for gene  in result_filtered['names'][group] if pd.notna(gene) and gene != '']
    })

    # top 10 per cluster from filtered results
    top10_filtered = (
        deg_filtered
        .groupby('cluster')
        .apply(lambda x: x.nlargest(10, 'score'))
        .reset_index(drop=True)
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # order clusters nicely
    cluster_order = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))

    # dotplot
    sc.pl.dotplot(
        adata,
        var_names=top10_filtered.groupby('cluster')['gene'].apply(list).to_dict(),  # grouped by cluster
        groupby='leiden',
        categories_order=cluster_order,
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(30, 10),
        show=False,
        save="_private_analysis.pdf"
    )

    # matrixplot
    sc.pl.matrixplot(
        adata,
        var_names=top10_filtered.groupby('cluster')['gene'].apply(list).to_dict(),
        groupby='leiden',
        categories_order=cluster_order,
        layer='lognorm',
        use_raw=False,
        standard_scale='var',
        cmap='RdBu_r',
        figsize=(30, 10),
        show=False,
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # alternative — dotplot (shows both expression and % cells expressing)
    sc.pl.dotplot(
        adata,
        var_names=top20_genes,
        groupby='leiden',
        layer='lognorm',
        use_raw=False,
        figsize=(40, 10),
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # alternative — matrixplot (cleaner for many clusters)
    sc.pl.matrixplot(
        adata,
        var_names= genes_by_cluster,
        groupby='leiden',
        layer='lognorm',
        use_raw=False,
        cmap='RdBu_r',
        standard_scale='var',
        figsize=(60, 20),
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # plot heatmap — equivalent to DoHeatmap in Seurat

    sc.pl.heatmap(
        adata,
        var_names=top20_genes,
        groupby='leiden',           # equivalent to grouping by cluster in Seurat
        layer='lognorm',            # use lognorm layer, same as Seurat's normalized data
        use_raw=False,
        show_gene_labels=True,
        standard_scale='var',
        figsize=(20, 10),           # match your pdf(width=40, height=50)
        save="_private_analysis.pdf"
    )


## Barcode exclusion and rebuilding a count object

The exclusion mechanism is retained; source selections and their biological rationale are withheld. `selected_clusters` is empty until supplied locally.

The fresh-count rebuild below applies only to the Matrix Market input route. With h5ad input, the preceding exclusion step operates on the loaded object and the matrix-reload cells are skipped.


In [ ]:
if RUN_PRIVATE_ANALYSIS and selected_clusters:
    # Store in adata.uns for provenance tracking
    removed_barcodes = adata.obs_names[adata.obs['leiden'].isin(selected_clusters)].tolist()
    removed_barcodes
    print(len(removed_barcodes)) 

    # Save to file
    import pandas as pd
    pd.Series(removed_barcodes).to_csv(OUTPUT_DIR / "excluded_barcodes.csv", index=False, header=False)

    adata = adata[~adata.obs['leiden'].isin(selected_clusters)].copy()
    adata
    adata.uns['removed_barcodes'] = removed_barcodes

    # Retrieve later
    removed = adata.uns['removed_barcodes']
    print(len(removed))


In [ ]:
if RUN_PRIVATE_ANALYSIS and selected_clusters and INPUT_FORMAT == "matrix":
    ##Directly opening clean adata without the removed barcodes
    # Reload fresh counts from disk
    counts_fresh = mmread(PRIVATE_DIR / "subset_counts.mtx").T.tocsr()
    obs_fresh = pd.read_csv(PRIVATE_DIR / "subset_metadata.csv", index_col=0)
    genes = pd.read_csv(PRIVATE_DIR / "subset_genes.csv")

    # Build full adata fresh
    adata_fresh = sc.AnnData(X=counts_fresh, obs=obs_fresh)
    adata_fresh.var_names = genes.iloc[:, 0].astype(str).values

    # Exclude removed barcodes
    mask = ~adata_fresh.obs_names.isin(removed_barcodes)
    adata_clean = adata_fresh[mask].copy()

    print(adata_clean.shape)        # verify cell count is reduced
    print(adata_clean.X.max())      # should be large integer = raw counts


In [ ]:
if RUN_PRIVATE_ANALYSIS and selected_clusters and INPUT_FORMAT == "matrix":
    adata_clean.layers['counts'] = adata_clean.X.copy()
    print("Counts layer saved. Max:", adata_clean.layers['counts'].max())


## Optional enrichment code

The source included an Enrichr query. The source feature list and results are withheld. This external-service step has a separate disabled flag and requires a list approved for submission; setting the general run flag alone will not call it.

In [ ]:
if RUN_PRIVATE_ANALYSIS and RUN_EXTERNAL_ENRICHMENT and approved_enrichment_genes:
    import gseapy as gp
    import pandas as pd
    import matplotlib.pyplot as plt

    enrichment_genes = approved_enrichment_genes

    # Query the source-selected public gene-set collection.
    databases = [
        #"KEGG_2021_Human",
        "GO_Biological_Process_2023",
        #"Reactome_2022"
    ]

    results = {}

    for db in databases:
        enr = gp.enrichr(
            gene_list   = enrichment_genes,
            gene_sets   = db,
            organism    = "human",
            outdir      = None,
            cutoff      = 0.05
        )
        results[db] = enr.results
        print(f"\n{db} — top hits:")
        print(enr.results[["Term", "P-value", "Adjusted P-value", "Genes"]].head(10))


In [ ]:
if RUN_PRIVATE_ANALYSIS and RUN_EXTERNAL_ENRICHMENT and approved_enrichment_genes:
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    for ax, db in zip(axes, databases):
        df = results[db]
        df = df[df["Adjusted P-value"] < 0.05].head(10)
    
        if df.empty:
            ax.set_title(f"{db}\nNo significant terms")
            continue
    
        df = df.sort_values("Adjusted P-value", ascending=True)
    
        ax.barh(
            df["Term"],
            -np.log10(df["Adjusted P-value"]),   # fixed here
            color="#1f77b4"
        )
        ax.set_xlabel("-log10(Adj P-value)")
        ax.set_title(db.replace("_", " "), fontsize=10)
        ax.invert_yaxis()

    plt.tight_layout()
    plt.show()


## Group-wise plotting and proportions

Only plotting and aggregation mechanics are retained. Private group ordering and classifications are obtained locally, not specified in this repository. Cell fractions are descriptive and are not donor-level tests.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import scanpy as sc
    import matplotlib.pyplot as plt

    group_order = adata.obs["group_id"].dropna().unique().tolist()

    

    # 1. Create Master Palette
    sc.pl.umap(adata, color='leiden', show=True, size=60)
    master_palette = dict(zip(adata.obs['leiden'].cat.categories, adata.uns['leiden_colors']))

    # 2. Setup figure
    ncols = 4
    nrows = (len(group_order) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
    axes = axes.flatten()

    # 3. Loop
    for i, group_id in enumerate(group_order):
        temp_col = f'plot_{group_id}'
        adata.obs[temp_col] = adata.obs['leiden'].astype(str)

        mask = adata.obs['group_id'] != group_id
        adata.obs.loc[mask, temp_col] = "Background"

        indices_bg = adata.obs.index[adata.obs[temp_col] == "Background"]
        indices_fg = adata.obs.index[adata.obs[temp_col] != "Background"]
        new_order = indices_bg.append(indices_fg)
        adata_subset = adata[new_order, :]

        local_palette = {**master_palette, 'Background': '#e0e0e0'}

        sc.pl.umap(
            adata_subset,
            color=temp_col,
            palette=local_palette,
            ax=axes[i],
            show=False,
            title="",
            frameon=False,
            size=60,
            legend_loc=None
        )

        axes[i].set_title(group_id, fontsize=15, fontweight='normal', pad=3)

        del adata.obs[temp_col]

    # 4. Clean up empty axes
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "private_plot.pdf", bbox_inches='tight')
    plt.show()


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors


    sample_order = adata.obs["group_id"].dropna().unique().tolist()

    # 1. Prepare data
    count_table = pd.crosstab(adata.obs['group_id'], adata.obs['leiden'])
    percent_table = count_table.div(count_table.sum(axis=1), axis=0) * 100

    df_long = percent_table.reset_index().melt(
        id_vars='group_id', 
        var_name='Cluster', 
        value_name='Percentage'
    )

    # 2. CREATE A MUCH LARGER FIGURE
    # height=7 and aspect=1.4 ensures the subplots are large enough to hold readable text
    g = sns.catplot(
        data=df_long, 
        x='group_id', 
        y='Percentage', 
        col='Cluster', 
        kind='bar', 
        col_wrap=5, 
        height=7, 
        aspect=1.4,
        order=sample_order,
        sharey=False
    )

    # 3. Formatting Loop
    axes = g.axes.flatten()
    n_plots = len(df_long['Cluster'].unique())
    cols = 5

    for i, ax in enumerate(axes):
        if i >= n_plots:
            ax.set_visible(False)
            continue

        # --- LOCAL COLORING ---
        heights = [bar.get_height() for bar in ax.patches]
        if heights and max(heights) > 0:
            local_norm = mcolors.Normalize(vmin=min(heights), vmax=max(heights))
            local_mapper = cm.ScalarMappable(norm=local_norm, cmap='viridis')
            for bar in ax.patches:
                bar.set_color(local_mapper.to_rgba(bar.get_height()))

        # --- FIX Y-AXIS (FIRST COLUMN) ---
        if i % cols == 0:
            ax.tick_params(labelleft=True, labelsize=12) # Increased font size
            ax.set_ylabel("Percentage (%)", fontsize=14, fontweight='bold')
        else:
            # Show numerical ticks even if not in first column because sharey=False
            ax.tick_params(labelleft=True, labelsize=10)

        # --- FIX X-AXIS (ALL PLOTS) ---
        # Since the previous versions were missing labels, we will force 
        # labels on EVERY plot to ensure you can see them on the top rows.
        ax.tick_params(labelbottom=True) 
        ax.set_xticklabels(sample_order, rotation=90, ha='right', fontsize=20) # Larger labels
        ax.set_xlabel("") 

    # 4. RADICAL SPACE ADJUSTMENTS TO PREVENT OVERLAP
    plt.subplots_adjust(
        wspace=0.4,    # Space between columns
        hspace=3,    # Massive vertical gap so labels don't hit titles below
        bottom=0.15,   
        left=0.1,      
        top=0.9
    )

    # Pad moves the title up so it doesn't clash with the labels above it
    g.set_titles("{col_name}", fontweight='bold', size=30, pad=40)

    # Save high-res to see detail
    plt.show()


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import os

    # ── 1. define orders ──────────────────────────────────────────────────────────

    sample_order = adata.obs["sample_id"].dropna().unique().tolist()


    annotation_order = adata.obs["annotation"].dropna().unique().tolist()

    colors = [
        '#1f77b4', '#ff7f0e', '#279e68', '#d62728', '#aa40fc', '#8c564b',
        '#e377c2', '#b5bd61', '#17becf', '#aec7e8', '#ffbb78', '#98df8a',
        '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#dbdb8d', '#9edae5',
        '#ad494a', '#8c6d31', '#FF34FF', '#763568', '#63FFAC', '#B79762',
        '#004D43', '#002dff', '#997D87', '#5A0007', '#809693', '#6A3A4C',
        '#1B4400', '#4FC601', '#3B5DFF', '#4A3B53', '#FF2F80', '#61615A',
        '#BA0900', '#6B7900', '#00C2A0', '#FFAA92', '#FF90C9', '#B903AA',
        '#D16100', '#DDEFFF', '#000035', '#7B4F4B', '#A1C299', '#300018',
        '#0AA6D8', '#013349', '#00846F', '#372101', '#FFB500', '#C2FFED',
        '#A079BF', '#CC0744', '#FFFF00', '#1CE6FF', '#FF34FF', '#FF4A46',
        '#008941', '#006FA6', '#A30059', '#FFDBE5', '#7A4900', '#0000A6',
        '#8FB0FF', '#2C3E50', '#4B0082', '#3E2723', '#1A237E', '#311B92'
    ]

    # ── 2. set ordered categoricals ───────────────────────────────────────────────

    adata.obs["annotation"] = pd.Categorical(
        adata.obs["annotation"],
        categories=annotation_order,
        ordered=True
    )

    adata.obs["sample_id"] = pd.Categorical(
        adata.obs["sample_id"],
        categories=sample_order,
        ordered=True
    )

    # ── 3. build proportions dataframe ────────────────────────────────────────────

    DF = adata.obs[["sample_id", "annotation"]].copy()

    DF = (
        DF.groupby(["sample_id", "annotation"], observed=True)
        .size()
        .reset_index(name="Count")
    )

    DF["total"]   = DF.groupby("sample_id")["Count"].transform("sum")
    DF["percent"] = DF["Count"] / DF["total"] * 100

    # ── 4. pivot ──────────────────────────────────────────────────────────────────

    pivot = DF.pivot_table(
        index="sample_id",
        columns="annotation",
        values="percent",
        fill_value=0
    )

    # apply orders
    sample_order_present = [s for s in sample_order if s in pivot.index]
    annotation_order_present = [e for e in annotation_order if e in pivot.columns]

    pivot = pivot.loc[sample_order_present, annotation_order_present]

    print(pivot.shape)
    print(pivot.head())

    # ── 5. plot ───────────────────────────────────────────────────────────────────

    fig, ax = plt.subplots(figsize=(14, 8))

    pivot.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=colors[:len(annotation_order_present)],
        edgecolor="none",
        width=0.8
    )

    ax.set_ylabel("Percent (%)", fontsize=13)
    ax.set_xlabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, va="top", fontsize=11)
    ax.legend(
        title="Annotation",
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        fontsize=11,
        title_fontsize=12
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.show()  
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "private_plot.pdf", bbox_inches="tight", dpi=300)
